In [7]:
!pip install Bio
from Bio import Entrez
import pandas as pd


# Read your upregulated genes
df = pd.read_csv("upregulated_genes.csv")

Entrez.email = "jamesonuh2020@gmail.com"
# Select the top DEGs
top_genes = df["Unnamed: 0"].tolist()
annotations = []

print(f"Annotating {len(top_genes)} genes using NCBI...")

for gene in top_genes:
    try:
        # Search NCBI Gene
        search = Entrez.esearch(
            db="gene",
            term=f"{gene}[Gene Name] AND Saccharomyces cerevisiae[Organism]",
            retmax=1
        )

        result = Entrez.read(search)
        search.close()

        if not result["IdList"]:
            print(f"No NCBI Gene ID found for {gene}")
            continue

        gene_id = result["IdList"][0]

        # Retrieve gene record
        fetch = Entrez.efetch(
            db="gene",
            id=gene_id,
            retmode="xml"
        )

        record = Entrez.read(fetch)
        fetch.close()

        info = record[0]

        # Gene description
        gene_data = info.get("Entrezgene_gene", {})
        gene_ref = gene_data.get("Gene-ref", {})

        if isinstance(gene_ref, list):
            gene_ref = gene_ref[0] if gene_ref else {}

        description = gene_ref.get("Gene-ref_desc", "N/A")

        # Function summary
        function = info.get("Entrezgene_summary", "N/A")

        # GO and KEGG terms
        go_terms = []
        kegg_terms = []

        sections = [
            info.get("Entrezgene_gene", {}),
            info.get("Entrezgene_locus", [])
        ]

        for section in sections:

            if isinstance(section, dict):
                section = [section]

            for item in section:

                for commentary in item.get("Gene-commentary", []):

                    label = commentary.get("Gene-commentary_label")
                    text = commentary.get("Gene-commentary_text")

                    if label == "GO" and text:
                        go_terms.append(text)

                    elif label == "KEGG" and text:
                        kegg_terms.append(text)

        # Store result
        annotations.append({
            "Gene": gene,
            "NCBI_Gene_ID": gene_id,
            "Description": description,
            "Function": function,
            "GO_Terms": "; ".join(go_terms) if go_terms else "N/A",
            "KEGG_Terms": "; ".join(kegg_terms) if kegg_terms else "N/A"
        })

        print(f"Annotated: {gene}")

    except Exception as e:
        print(f"Error processing {gene}: {e}")


# Convert annotations to DataFrame
annotations_df = pd.DataFrame(annotations)

# Save results
annotations_df.to_csv("NCBI_gene_annotations.csv", index=False)

print("\nAnnotation complete!")
print("Results saved to NCBI_gene_annotations.csv")

Annotating 1176 genes using NCBI...
Annotated: TFC3
Annotated: VPS8
Annotated: SSA1
Annotated: ERP2
Annotated: FUN14
Annotated: PSK1
Annotated: FRT2
Annotated: GIP4
Annotated: FUN19
Annotated: ACS1
Annotated: GPB2
Annotated: YAL064W-B
Annotated: PAU8
Annotated: SWD1
Annotated: YAR009C
Annotated: PAU7
Annotated: YAR023C
Annotated: YAR029W
Annotated: PHO11
Annotated: YBL005W-B
Annotated: SLA1
Annotated: ACH1
Annotated: FUS3
Annotated: PEP1
Annotated: YBL029C-A
Annotated: RIB1
Annotated: YBL039W-B
Annotated: ERD2
Annotated: ECM13
Annotated: SEC17
Annotated: PRX1
Annotated: SSA3
Annotated: ATG8
Annotated: YBL086C
Annotated: SCS22
Annotated: YBL100W-B
Annotated: ECM21
Annotated: NTH2
Annotated: RCR1
Annotated: UGA2
Annotated: YBR012W-B
Annotated: YBR013C
Annotated: GRX7
Annotated: YBR016W
Annotated: GAL7
Annotated: SCO2
Annotated: ETR1
Annotated: GIP1
Annotated: ZTA1
Annotated: FMP23
Annotated: REG2
Annotated: RFS1
Annotated: YBR053C
Annotated: YRO2
Annotated: YBR056W
Annotated: YBR056W-A
A

In [9]:
result=pd.read_csv("NCBI_gene_annotations.csv")

In [10]:
result.head()

,Gene,NCBI_Gene_ID,Description,Function,GO_Terms,KEGG_Terms
0,TFC3,851262,NaN,"Contributes to DNA binding activity, bending; ...",NaN,NaN
1,VPS8,851261,NaN,Enables GTPase binding activity and protein-me...,NaN,NaN
2,SSA1,851259,NaN,Enables ATP hydrolysis activity; tRNA binding ...,NaN,NaN
3,ERP2,851226,NaN,Involved in endoplasmic reticulum to Golgi ves...,NaN,NaN
4,FUN14,851225,NaN,Involved in mitochondrion organization and pho...,NaN,NaN


In [11]:
import os

old_filename = "NCBI_gene_annotations.csv"
new_filename = "annotations.csv"

if os.path.exists(old_filename):
    os.rename(old_filename, new_filename)
    print(f"File renamed from {old_filename} to {new_filename}")
else:
    print(f"File {old_filename} does not exist.")


File renamed from NCBI_gene_annotations.csv to annotations.csv
